In [27]:
from pprint import pprint
import pathlib
import csv
import os
from utils import *
import csv
import contractions
import shutil
import emoji
from emot.emo_unicode import EMOTICONS_EMO
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /home/lenovo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/lenovo/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /home/lenovo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [22]:
# create preprocessed dir if not exists
pathlib.Path(preprocessed_data_path).mkdir(parents=True, exist_ok=True)

# copy raw reviews for preprocessing
shutil.copytree(raw_data_path, preprocessed_data_path, dirs_exist_ok=True)

def preprocess_data(process_fn):
    for name in os.listdir(preprocessed_data_path):
        path = os.path.join(preprocessed_data_path, name)

        with open(path, newline="") as f:
            reader = csv.DictReader(f)
            data = list(reader)

            for review in data:
                text = process_fn(review["content"])
                review["content"] = text

            write_path = os.path.join(preprocessed_data_path, name)
            file = open(write_path, "w")
            output_csv(data, file)

            file.close()

In [23]:
contractions_dict = {
    "i'mma": "i will"
}

def expand_contractions(content):
    expanded_words = []
    content_words = content.split(" ")
    for word in content_words:
        word_new = ""
        if word not in contractions_dict.keys():
            word_new = contractions.fix(word)
        else:
            word_new = contractions_dict[word]
        expanded_words.append(word_new)
    return " ".join(expanded_words)

preprocess_data(expand_contractions)

In [24]:
def remove_emojis(content):
    return emoji.replace_emoji(content, replace="")

preprocess_data(remove_emojis)

In [25]:
emoticons_dict_custom = EMOTICONS_EMO
emoticons_dict_custom["¯\\_(ツ)_/¯"] = "Shrug"

emoticon_regex = re.compile(
    "|".join(map(re.escape, emoticons_dict_custom.keys()))
)

def remove_emoticons(content):
    return re.sub(emoticon_regex, "", content)
preprocess_data(remove_emoticons)

In [26]:
def remove_stopwords(content):
    content_words = content.split(" ")
    filtered = [w for w in content_words if w not in STOPWORDS]

    filtered_initial_text = " ".join(filtered)

    # do it again with nltk
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(filtered_initial_text.lower())

    filtered_tokens = [word for word in tokens if word not in stop_words]

    return " ".join(filtered_tokens)

preprocess_data(remove_stopwords)

LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/home/lenovo/nltk_data'
    - '/home/lenovo/Documents/School/Sem6/PBA/pba-task-1a-individual/.env/nltk_data'
    - '/home/lenovo/Documents/School/Sem6/PBA/pba-task-1a-individual/.env/share/nltk_data'
    - '/home/lenovo/Documents/School/Sem6/PBA/pba-task-1a-individual/.env/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
